# Figure 3A2 / 3B2 — grit score panels

Rebuild of the main-figure grit bar plots so that they comply with the journal's data-presentation
policy: **every bar carries the individual data points**, and all plot elements plus the precise
*n* are defined in an auto-generated legend.

Two figures, each 2 rows (HCT116, HT29) x 4 columns (SN-38, binimetinib, abemaciclib, controls):

| figure | data cut | panel in `Figure3-01.tif` |
|---|---|---|
| **3A2** | `MIP` | c |
| **3B2** | `aggregates` (scAgg) | d |

Each figure is drawn at the size it occupies in the assembled figure (measured from
`Figure3-01.tif`: panel c is 57.2 x 44 mm at 300 dpi), so the SVG/PDF drops into the layout
at 100% - no rescaling, and therefore no font-size drift.

Text is Arial and stays editable in Illustrator, following the convention used in
`3_Figure4/PairwiseCorrelations/Pairwise_related_figures/`: `svg.fonttype='none'`
(SVG text stays `<text>` in Arial, not outlines) and `pdf.fonttype=42`.

Grit is read per well from the pre-computed `1_Data/results/grit_data_{data_type}_{cell_line}.parquet`
files (written by `3_GritScores.ipynb`), so the values are identical to the previously published panels.

Outputs (in `3_Figure3/GritScores/result-images/`): `Figure3A2_grit_MIP.{svg,pdf,png}`,
`Figure3B2_grit_scAgg.{svg,pdf,png}`, `Figure3A2B2_n_per_condition.csv`, `Figure3A2B2_legend.txt`.

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline
import seaborn as sns; sns.set_style("white")

# Run from the project root so all relative paths match the other notebooks
if os.path.basename(os.getcwd()) != 'spher_colo52_v1':
    pass  # was os.chdir; the bootstrap above handles paths


In [ ]:
# Plotting parameters - same convention as 3_Figure4/.../Pairwise_related_figures:
# Arial everywhere, text stays editable text (not outlines) in the SVG and the PDF.
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'svg.fonttype': 'none',    # SVG keeps <text> elements -> real Arial in Illustrator
    'pdf.fonttype': 42,        # TrueType, editable text in the PDF
    'ps.fonttype': 42,
    'font.family': 'sans-serif',
    # Liberation Sans is metrically identical to Arial and is used as the local stand-in
    # when Arial is not installed on the server, so the layout matches Arial exactly.
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 6,
    'axes.titlesize': 6.5,
    'axes.labelsize': 6,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5.5,
    'legend.fontsize': 5.5,
})
dpi = 300

DataDir = str(profiles("exp1_main", "")) + "/"
ImagesOut = str(figdir("Fig3")) + "/"
os.makedirs(ImagesOut, exist_ok=True)


def save(fig, stem):
    # No bbox_inches='tight' - the figure is already drawn at its final print size,
    # so the files come out at exactly PANEL_W_MM x PANEL_H_MM and drop into the layout at 100%.
    for ext in ('png', 'svg', 'pdf'):
        fig.savefig('{}{}.{}'.format(ImagesOut, stem, ext))
    # matplotlib writes the whole font stack into the SVG; Illustrator is happiest with a
    # single family name, so declare plain Arial.
    path = '{}{}.svg'.format(ImagesOut, stem)
    with open(path) as fh:
        svg = fh.read()
    svg = svg.replace("'Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans', sans-serif", 'Arial')
    with open(path, 'w') as fh:
        fh.write(svg)
    print('wrote {}{}.svg (+ .pdf, .png)'.format(ImagesOut, stem))

## Configuration — everything that controls the panels lives here

In [ ]:
# data cut -> panel letter in the manuscript figure
FIGURES = {'MIP': '3c', 'aggregates': '3d'}
# short label used in the file names
FIG_DATA_LABEL = {'MIP': 'MIP', 'aggregates': 'scAgg'}

CELL_LINES = ['HCT116', 'HT29']          # one row per cell line
CONTROLS = ['etop', 'fenb', 'stau', 'dmso']
PANELS = ['SN-38', 'Binim', 'abema', 'controls']   # one column per entry

# Complete, decapitalised drug names (Metadata_name is the 5-character key in the parquet)
DRUG_LABELS = {
    'SN-38': 'SN-38',
    'Binim': 'binimetinib',
    'abema': 'abemaciclib',
    'etop':  'etoposide 2.5',
    'fenb':  'fenbendazole 2.5',
    'stau':  'staurosporine 0.1',
    'dmso':  'DMSO 0.1',
}

GRIT_THRESHOLD = 1.96      # dashed reference line
# the 'grit = 1.96' key is drawn once for the pair, under 3B2 (as in the assembled figure)
THRESHOLD_LEGEND_ON = ['3d']

# Panel size, measured from the assembled figure (Figure3-01.tif, 300 dpi): panel c spans
# x 1005-1680, y 40-559 px -> 57.2 x 44.0 mm. Drawn at that size (a little taller to fit the
# threshold legend) the SVG drops into the layout at 100%.
PANEL_W_MM, PANEL_H_MM = 57.2, 46.0
FIGSIZE = (PANEL_W_MM / 25.4, PANEL_H_MM / 25.4)

# fixed margins (no tight_layout) so both figures have identical axes positions
MARGINS = dict(left=0.085, right=0.915, top=0.90, bottom=0.30, wspace=0.45, hspace=0.28)
print('figure size: {:.2f} x {:.2f} inches ({} x {} mm)'.format(*FIGSIZE, PANEL_W_MM, PANEL_H_MM))

## Load the per-well grit scores

In [ ]:
KEEP = ['Metadata_name', 'Metadata_cmpdname', 'Metadata_cmpd_conc',
        'Metadata_conc_step', 'Metadata_PlateWell', 'Metadata_grit']


def load_grit(data_type, cell_line):
    # one row per well, only the metadata we need (not the ~470 features)
    df = pd.read_parquet(
        '{}grit_data_{}_{}.parquet'.format(DataDir, data_type, cell_line), columns=KEEP)
    df = df.rename(columns={'Metadata_grit': 'grit'})
    df['data_type'] = data_type
    df['cell_line'] = cell_line
    return df


grit = pd.concat([load_grit(dt, cl) for dt in FIGURES for cl in CELL_LINES],
                 ignore_index=True)

# one dot = one well: check we did not pick up duplicated wells
assert not grit.duplicated(['data_type', 'cell_line', 'Metadata_PlateWell']).any()

# keep only the compounds shown in the main figure
grit = grit[grit['Metadata_name'].isin(['SN-38', 'Binim', 'abema'] + CONTROLS)].copy()

grit.groupby(['data_type', 'cell_line']).size()

In [ ]:
# Shared y-limits across both figures, wide enough that no dot and no error bar is clipped
_stats = grit.groupby(['data_type', 'cell_line', 'Metadata_name',
                       'Metadata_cmpd_conc'])['grit'].agg(['mean', 'std'])
_hi = max(grit['grit'].max(), (_stats['mean'] + _stats['std'].fillna(0)).max())
_lo = min(grit['grit'].min(), (_stats['mean'] - _stats['std'].fillna(0)).min())
YLIM = (np.floor((_lo - 0.2) * 10) / 10, np.ceil((_hi + 0.2) * 10) / 10)
print('y-limits:', YLIM)

## Plotting helpers

In [ ]:
def conc_label(c):
    # 0.03 -> '0.03', 10.0 -> '10.0' (as in the published panels)
    return '{:g}'.format(c) if c < 1 else '{:.1f}'.format(c)


def draw_panel(ax, data, compound, show_xticklabels):
    # One compound column: bars = mean, error bars = SD, dots = individual wells
    if compound == 'controls':
        d = data[data['Metadata_name'].isin(CONTROLS)].copy()
        order = CONTROLS
        d['xcat'] = pd.Categorical(d['Metadata_name'], categories=order, ordered=True)
        d = d.sort_values('xcat')
        palette = dict(zip(order, plt.cm.Blues(np.linspace(0.3, 0.9, len(order)))))
        labels = [DRUG_LABELS[c] for c in order]
        title, xlabel = 'controls', ''
    else:
        d = data[data['Metadata_name'] == compound].copy()
        order = sorted(d['Metadata_cmpd_conc'].unique())
        d['xcat'] = pd.Categorical(d['Metadata_cmpd_conc'], categories=order, ordered=True)
        # numeric hue over 'Blues_d' - reproduces the shading of the published panels
        # (lightest -> near-black for the top dose)
        palette = 'Blues_d'
        labels = [conc_label(c) for c in order]
        title, xlabel = DRUG_LABELS[compound], 'µM'

    hue = 'Metadata_name' if compound == 'controls' else 'Metadata_conc_step'
    kw = {'hue_order': order} if compound == 'controls' else {}
    sns.barplot(data=d, x='xcat', y='grit', hue=hue, order=order,
                palette=palette, dodge=False, legend=False,
                estimator='mean', errorbar='sd',
                err_kws={'linewidth': 0.6, 'color': 'black'},
                edgecolor='none', linewidth=0,      # flat fills, no bar outline
                width=0.85, ax=ax, zorder=2, **kw)
    sns.stripplot(data=d, x='xcat', y='grit', order=order, dodge=False, jitter=0.14,
                  size=1.2, color='black', linewidth=0, alpha=0.9,
                  ax=ax, zorder=3, legend=False)

    ax.axhline(y=GRIT_THRESHOLD, color='black', linestyle=(0, (3, 2)), linewidth=0.7,
               alpha=0.9, zorder=4)
    ax.set_facecolor('w')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for side in ['bottom', 'left']:
        ax.spines[side].set_color('grey')
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis='both', length=0, pad=1.0)
    ax.set_title(title, pad=1.5)
    ax.set_yticks([0, 2, 4, 6])
    ax.set_ylim(YLIM)
    ax.set_xlabel('')
    ax.set_ylabel('')

    ax.set_xticks(range(len(order)))
    if show_xticklabels:
        ax.set_xticklabels(labels, rotation=45, ha='right', rotation_mode='anchor')
        ax.set_xlabel(xlabel, labelpad=1)
    else:
        ax.set_xticklabels([])
    return d

In [ ]:
def plot_grit_figure(data_type, save_files=True):
    # 2 rows (cell lines) x 4 columns (compounds) for one data cut
    sub = grit[grit['data_type'] == data_type]
    fig, axes = plt.subplots(len(CELL_LINES), len(PANELS), figsize=FIGSIZE, sharey=True)

    for r, cell_line in enumerate(CELL_LINES):
        for c, compound in enumerate(PANELS):
            ax = axes[r, c]
            draw_panel(ax, sub[sub['cell_line'] == cell_line], compound,
                       show_xticklabels=(r == len(CELL_LINES) - 1))
            if r > 0:                      # column titles only on the top row
                ax.set_title('')
            if c == 0:
                ax.set_ylabel('grit', labelpad=1)
        # cell line label on the right, as in the manuscript figure
        axes[r, -1].text(1.05, 0.5, cell_line, transform=axes[r, -1].transAxes,
                         rotation=270, va='center', ha='left', fontsize=7)

    fig.subplots_adjust(**MARGINS)
    if FIGURES[data_type] in THRESHOLD_LEGEND_ON:
        fig.legend([Line2D([0], [0], color='black', linestyle=(0, (3, 2)), linewidth=0.7)],
                   ['grit = {}'.format(GRIT_THRESHOLD)], loc='lower right',
                   bbox_to_anchor=(1.0, 0.0), frameon=False, handlelength=1.8,
                   handletextpad=0.3, borderaxespad=0.2)

    if save_files:
        save(fig, 'Figure{}_grit_{}'.format(FIGURES[data_type], FIG_DATA_LABEL[data_type]))
        save_panel(fig, 'Fig' + FIGURES[data_type], data=sub,
                   caption=f'Grit scores per compound, {FIG_DATA_LABEL[data_type]}',
                   notebook='analysis/3_Figure3/3_GritScores_Fig3cd.ipynb')
    return fig

## Figure 3A2 — MIP (panel c)

In [ ]:
fig_3c = plot_grit_figure('MIP')
plt.show()

## Figure 3B2 — scAgg / aggregates (panel d)

In [ ]:
fig_3d = plot_grit_figure('aggregates')
plt.show()

## Precise n per condition (for the figure legend / supplementary table)

In [ ]:
n_table = (
    grit.groupby(['data_type', 'cell_line', 'Metadata_name', 'Metadata_cmpd_conc'])['grit']
        .agg(n='size', mean='mean', sd='std')
        .reset_index()
)
n_table['figure'] = n_table['data_type'].map(FIGURES)
n_table['compound'] = n_table['Metadata_name'].map(DRUG_LABELS)
n_table = n_table.rename(columns={'Metadata_cmpd_conc': 'conc_uM'})
n_table = n_table[['figure', 'data_type', 'cell_line', 'compound', 'Metadata_name',
                   'conc_uM', 'n', 'mean', 'sd']].round(3)
n_table = n_table.sort_values(['figure', 'cell_line', 'Metadata_name', 'conc_uM'])

n_table.to_csv('{}Figure3cd_n_per_condition.csv'.format(ImagesOut), index=False)
print('n range, compound doses:',
      n_table.loc[n_table['Metadata_name'] != 'dmso', 'n'].min(), '-',
      n_table.loc[n_table['Metadata_name'] != 'dmso', 'n'].max())
n_table

## Auto-generated legend text (elements + exact n)

In [ ]:
def legend_text(data_type):
    tbl = n_table[n_table['data_type'] == data_type]
    lines = [
        'Figure {}. Grit scores per well, {} profiles.'.format(
            FIGURES[data_type], FIG_DATA_LABEL[data_type]),
        'Bars, mean grit across replicate wells; error bars, +/- 1 s.d.; '
        'dots, individual wells (one dot per well, all wells shown); '
        'dashed line, grit = {} (threshold for reproducible morphological activity). '
        'Rows, cell line; x axis, compound concentration in uM.'.format(
            GRIT_THRESHOLD),
        'Exact n (wells per condition):',
    ]
    for cell_line in CELL_LINES:
        parts = []
        for key in ['SN-38', 'Binim', 'abema'] + CONTROLS:
            rows = tbl[(tbl['cell_line'] == cell_line) & (tbl['Metadata_name'] == key)]
            if rows.empty:
                continue
            if key in CONTROLS:
                parts.append('{} n = {}'.format(DRUG_LABELS[key], int(rows['n'].iloc[0])))
            else:
                doses = '; '.join('{} uM, n = {}'.format(conc_label(c), int(n))
                                  for c, n in zip(rows['conc_uM'], rows['n']))
                parts.append('{} ({})'.format(DRUG_LABELS[key], doses))
        lines.append('  {}: {}.'.format(cell_line, '; '.join(parts)))
    return '\n'.join(lines)


legend_blocks = '\n\n'.join(legend_text(dt) for dt in FIGURES)
with open('{}Figure3cd_legend.txt'.format(ImagesOut), 'w') as fh:
    fh.write(legend_blocks + '\n')
print(legend_blocks)

## Font check

Confirms which font file the text was actually drawn with. On a machine with Arial installed
this resolves to Arial; here it falls back to Liberation Sans, which has identical metrics,
and the SVG still declares `Arial` because `svg.fonttype='none'` writes the requested family.

In [ ]:
import matplotlib.font_manager as fm

resolved = fm.findfont(fm.FontProperties(family=plt.rcParams['font.sans-serif']))
print('font used for layout:', resolved)

svg = open('{}Figure3c_grit_MIP.svg'.format(ImagesOut)).read()
print('font-family declared in the SVG:',
      sorted({s.split(':')[1].strip() for s in svg.split(';') if s.strip().startswith('font-family')}))
print('SVG <text> elements (editable):', svg.count('<text'))

from PIL import Image
for stem in ['Figure3c_grit_MIP', 'Figure3d_grit_scAgg']:
    w, h = Image.open('{}{}.png'.format(ImagesOut, stem)).size
    print('{}.png: {} x {} px = {:.1f} x {:.1f} mm at 300 dpi'.format(
        stem, w, h, w / 300 * 25.4, h / 300 * 25.4))